# 🎙️ Scribe STT — Whisper large-v3 sur GPU gratuit (Colab)

1. Menu **Exécution → Modifier le type d'exécution → T4 GPU**.
2. Exécute les 3 cellules dans l'ordre (▶).
3. Copie l'URL affichée dans `scribe-v2/backend/.env` (`STT_ENDPOINT_URL`).

In [ ]:
# 1) Dépendances + tunnel cloudflared (aucun compte requis)
!pip -q install faster-whisper fastapi "uvicorn[standard]" python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('OK')

In [ ]:
# 2) Le micro-service
app_code = r'''
import os, tempfile
from functools import lru_cache
from fastapi import FastAPI, File, Form, UploadFile
app = FastAPI()
@lru_cache(maxsize=1)
def get_model():
    from faster_whisper import WhisperModel
    return WhisperModel('large-v3', device='cuda', compute_type='float16')
@app.get('/health')
def health():
    return {'status': 'ok'}
@app.post('/v1/audio/transcriptions')
async def transcriptions(file: UploadFile = File(...), model: str = Form(default='large-v3'), language: str = Form(default='fr')):
    suffix = os.path.splitext(file.filename or 'a.webm')[1] or '.webm'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    with tmp as fh: fh.write(await file.read())
    try:
        segments, info = get_model().transcribe(tmp.name, language=language, beam_size=5, vad_filter=True)
        text = ' '.join(s.text.strip() for s in segments).strip()
    finally:
        os.unlink(tmp.name)
    return {'text': text}
'''
open('stt_app.py', 'w').write(app_code)
print('service écrit')

In [ ]:
# 3) Démarre le service (attente + log d'erreur) puis le tunnel
import subprocess, time, urllib.request, re
subprocess.Popen(['uvicorn', 'stt_app:app', '--host', '0.0.0.0', '--port', '9000'],
                 stdout=open('uvicorn.log', 'w'), stderr=subprocess.STDOUT)
ok = False
for _ in range(90):
    try:
        if urllib.request.urlopen('http://localhost:9000/health', timeout=2).status == 200:
            ok = True; break
    except Exception:
        time.sleep(1)
if not ok:
    print('uvicorn KO. Log :\n'); print(open('uvicorn.log').read())
else:
    print('Service up. Tunnel...')
    tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:9000'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tun.stdout:
        m = re.search(r'https://[-\w.]+trycloudflare\.com', line)
        if m:
            print('\n=== URL pour .env (STT_ENDPOINT_URL) ==='); print(' ', m.group(0)); break

Dans `scribe-v2/backend/.env` : `STT_ENDPOINT_URL=https://xxxx.trycloudflare.com` puis relance le back-end.
Garde cet onglet ouvert (sinon l'URL meurt).